# Course 8 · Data Visualization with Python
Matplotlib → Seaborn → Folium maps → Plotly → Dash. Every line has a `#` comment.
**Choose the chart by the question:** trend → line · compare → bar · distribution → histogram/box · relationship → scatter · location → map.

## Step 1 · Load the clean data

In [ ]:
# os lets Python talk to the operating system (files, folders).
import os
# If the practice data file is not in this folder yet...
if not os.path.exists("applications.csv"):
    # ...run make_data.py to create applications.csv and centers.csv.
    %run make_data.py
# pandas = tables (nickname pd); numpy = fast math (nickname np). These nicknames are standard.
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns   # plotting libraries (standard nicknames)

# Load and clean the data in one chain (same steps as notebook 07):
df = (pd.read_csv("applications.csv", parse_dates=["submitted"])   # load; read dates as dates
        .drop_duplicates()                                         # remove duplicate rows
        .assign(channel=lambda d: d.channel.str.strip().str.lower(),   # tidy the labels
                age=lambda d: d.age.fillna(d.age.median()),            # fill missing ages
                month=lambda d: d.submitted.dt.month)                  # month number 1-12
        .merge(pd.read_csv("centers.csv"), on="center_id"))        # add region, lat, lon, staff
print(df.shape)                                                    # (rows, columns)

## Step 2 · Matplotlib: the two styles
**Scripting** (`plt.` functions) for quick plots; **object-oriented** (`fig, ax`) for full control.

In [ ]:
# Count applications per month: group by month, count app_id in each group.
monthly = df.groupby("month")["app_id"].count()

# SCRIPTING STYLE: plt.* functions act on "the current chart".
plt.plot(monthly.index, monthly.values, marker="o")   # x = months, y = counts, dot on each point
plt.title("Applications per month")
plt.xlabel("Month")
plt.ylabel("Applications")
plt.show()                                            # draw the chart

In [ ]:
# Average processing days per region, sorted so the bar chart reads smallest to largest.
by_region = df.groupby("region")["processing_days"].mean().sort_values()

# OBJECT-ORIENTED STYLE: fig = the whole picture; axes = a 2x2 grid of individual charts.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
# pandas can plot directly; ax= says which panel to draw in (row, column).
monthly.plot(kind="line", marker="o", ax=axes[0, 0], title="Applications per month")
by_region.plot(kind="barh", color="steelblue", ax=axes[0, 1], title="Avg processing days by region")
df["processing_days"].plot(kind="hist", bins=30, ax=axes[1, 0], title="Processing days")  # 30 bars
df.boxplot(column="processing_days", by="channel", ax=axes[1, 1])   # one box per channel
axes[1, 1].set_title("By channel")
fig.suptitle("")                                  # remove the automatic title boxplot adds
for ax in axes.flat:                              # .flat loops over all 4 panels
    ax.spines[["top", "right"]].set_visible(False)   # hide top/right borders for a cleaner look
plt.tight_layout()                                # space panels so labels don't overlap
plt.savefig("overview.png", dpi=150)              # save as an image file (dpi = sharpness)
plt.show()

## Step 3 · Area, pie and bubble plots

In [ ]:
# crosstab counts rows for each month x channel combination -> a table for the area chart.
ct = pd.crosstab(df.month, df.channel)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))     # 1 row, 3 charts

# AREA: stacked=True piles channels on top of each other to show the total and the mix.
ct.plot(kind="area", stacked=True, alpha=.6, ax=ax[0], title="Channel mix over time")

# PIE: autopct shows each slice's percentage. Use sparingly: bars compare sizes more accurately.
df.channel.value_counts().plot(kind="pie", autopct="%1.0f%%", ax=ax[1], title="Channel share")
ax[1].set_ylabel("")                              # remove the default y label on a pie

# BUBBLE: a scatter where marker size (s=) shows a third variable (application volume).
cs = df.groupby("center_id").agg(staff=("staff", "first"),           # one row per center
                                 days=("processing_days", "mean"),
                                 apps=("app_id", "count"))
ax[2].scatter(cs.staff, cs.days, s=cs.apps, alpha=.5)
ax[2].set(title="Staff vs days (bubble = volume)", xlabel="Staff", ylabel="Avg days")
plt.tight_layout()
plt.show()

## Step 4 · Seaborn statistical plots

In [ ]:
sns.set_theme(style="whitegrid")                  # a clean default style for all Seaborn charts
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# countplot: bars of counts per category; hue splits each bar by a second variable.
sns.countplot(data=df, x="channel", hue="rejected", ax=ax[0, 0])
# histplot with kde=True adds a smooth density curve; hue overlays two groups.
sns.histplot(data=df, x="processing_days", hue="docs_complete", kde=True, ax=ax[0, 1])
# regplot: scatter + fitted regression line (with a shaded confidence band).
sns.regplot(data=df.sample(800, random_state=0), x="biometric_quality", y="processing_days",
            scatter_kws={"alpha": .3}, ax=ax[1, 0])
# heatmap of correlations: annot=True writes the numbers; center=0 makes 0 white, +red, -blue.
cols = ["age", "staff", "biometric_quality", "docs_complete", "processing_days"]
sns.heatmap(df[cols].corr(), annot=True, cmap="RdBu_r", center=0, ax=ax[1, 1])
plt.tight_layout()
plt.show()

## Step 5 · Folium map of centers
Red = slower than the median center; circle size = number of applications. Locations are random practice values.

In [ ]:
import folium                                    # interactive web maps
from folium.plugins import MarkerCluster         # groups nearby markers when zoomed out

# One row per center with its location, number of applications and average days.
cs = (df.groupby(["center_id", "lat", "lon", "region"])
        .agg(apps=("app_id", "count"), days=("processing_days", "mean"))
        .reset_index())                          # turn the group labels back into columns

m = folium.Map(location=[9.0, 40.0], zoom_start=6)   # base map centred on Ethiopia
cluster = MarkerCluster().add_to(m)              # a cluster layer on the map
for _, r in cs.iterrows():                       # loop over rows; r is one center
    color = "red" if r.days > cs.days.median() else "green"
    folium.CircleMarker(
        [r.lat, r.lon],                          # position
        radius=r.apps / 15,                      # bigger circle = more applications
        color=color, fill=True,
        popup=f"Center {r.center_id} ({r.region}): {r.apps} apps, {r.days:.1f} days",  # click text
    ).add_to(cluster)
m.save("centers_map.html")                       # save as a web page you can open or share
m                                                # display the map in the notebook

## Step 6 · Interactive charts with Plotly Express

In [ ]:
import plotly.express as px                      # high-level, one-line interactive charts

# Scatter: one dot per center, coloured by region, sized by volume; hover shows center_id.
px.scatter(cs, x="apps", y="days", color="region", size="apps", hover_data=["center_id"],
           title="Centers: volume vs processing time").show()

# Line chart needs a tidy table: one row per month x channel. as_index=False keeps them as columns.
trend = df.groupby(["month", "channel"], as_index=False)["processing_days"].mean()
px.line(trend, x="month", y="processing_days", color="channel", markers=True,
        title="Avg processing days by month and channel").show()

## Step 7 · Dash dashboard
Run the next cell to write `app.py`. Then open **File → New → Terminal** and run `python app.py`, and open http://127.0.0.1:8050.
*On Anaconda's cloud notebooks the app's local link may not open. Run Dash on your laptop's local JupyterLab instead.*

In [ ]:
%%writefile app.py
# %%writefile saves this cell's text as a file instead of running it.
import pandas as pd, plotly.express as px
from dash import Dash, dcc, html, Input, Output   # Dash = app; dcc = controls/graphs; html = page elements

# Load and lightly clean the data once, when the app starts.
df = pd.read_csv("applications.csv").merge(pd.read_csv("centers.csv"), on="center_id")
df["channel"] = df["channel"].str.strip().str.lower()

app = Dash(__name__)                             # create the app
# LAYOUT: what the page contains, top to bottom.
app.layout = html.Div([
    html.H2("ID Registration Dashboard"),                                  # a heading
    dcc.Dropdown(sorted(df.region.unique()), "Oromia", id="region", clearable=False),  # options, default
    html.Div([dcc.Graph(id="by-channel"), dcc.Graph(id="dist")],          # two empty charts...
             style={"display": "flex"}),                                   # ...side by side
])

# CALLBACK: whenever the dropdown (Input) changes, run update() and send its results to the two graphs (Outputs).
@app.callback(Output("by-channel", "figure"), Output("dist", "figure"), Input("region", "value"))
def update(region):
    d = df[df.region == region]                                            # rows for the chosen region
    bar = px.bar(d.groupby("channel", as_index=False).processing_days.mean(),
                 x="channel", y="processing_days", title=f"Avg days by channel: {region}")
    hist = px.histogram(d, x="processing_days", color="rejected", title="Distribution")
    return bar, hist                                                       # same order as the Outputs

if __name__ == "__main__":                      # only start the server when run as a script
    app.run(debug=True, port=8050)

## ✍️ Your turn
1. Plot rejection rate by age group as a bar chart with a title and labelled axes.
2. Add a second dropdown (channel) to `app.py` and use it in the callback.

In [ ]:
# your answer here